# 01. Data Integrity and Initial Visual Inspection

This notebook performs an initial exploratory data analysis pass and data integrity audit on the Paddy Doctor dataset located at `data/raw/paddy-disease-classification/train_images/`. Its purpose is to verify image file integrity, check for corrupted or anomalous files, inspect image dimension consistency, and visually examine sample images across all 10 disease and normal classes.

## 1. Setup
Import required libraries and resolve paths to the dataset directory.

In [ ]:
import os
import random
from pathlib import Path
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

# Configure matplotlib inline display
%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 4)

# Fix random seed for reproducible sampling
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Resolve dataset directory (supports running from either notebooks/ or project root)
possible_paths = [
    Path("../data/raw/paddy-disease-classification/train_images"),
    Path("data/raw/paddy-disease-classification/train_images"),
]

DATA_DIR = None
for p in possible_paths:
    if p.exists() and p.is_dir():
        DATA_DIR = p.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate 'train_images/'. Please check data/raw/paddy-disease-classification/train_images"
    )

print(f"Dataset directory located at: {DATA_DIR}")

## 2. Integrity Check
Walk through all 10 class subfolders under `train_images/`. For every image file:
- Check for zero-byte size and unexpected file extensions
- Open and verify image data integrity using Pillow (`PIL.Image`)
- Record format, dimensions (width x height), and file size
- Flag any unreadable or corrupted files

In [ ]:
EXTENSION_TO_FORMAT = {
    ".jpg": "JPEG",
    ".jpeg": "JPEG",
    ".png": "PNG",
}

records = []
flagged_files = []

# Discover class subfolders
class_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Discovered {len(class_dirs)} class folders:")
for d in class_dirs:
    print(f"  - {d.name}")

print("\nScanning image files for integrity...")
for class_dir in class_dirs:
    class_name = class_dir.name
    for file_path in class_dir.iterdir():
        if file_path.is_dir():
            continue

        file_size_bytes = file_path.stat().st_size
        suffix = file_path.suffix.lower()

        # Flag zero-byte files
        if file_size_bytes == 0:
            flagged_files.append({
                "path": str(file_path),
                "class": class_name,
                "issue": "Zero-byte file"
            })
            continue

        # Verify and decode image
        try:
            with Image.open(file_path) as img:
                img.verify()  # Check header and file integrity
            with Image.open(file_path) as img:
                img.load()    # Decode pixel content to ensure no payload corruption
                width, height = img.size
                img_format = img.format
                img_mode = img.mode

            # Compare file extension against real decoded format
            expected_format = EXTENSION_TO_FORMAT.get(suffix)
            if expected_format != img_format:
                print(f"Extension mismatch: {file_path} (claimed: {suffix}, actual: {img_format})")

            records.append({
                "file_path": str(file_path),
                "filename": file_path.name,
                "class_name": class_name,
                "format": img_format,
                "mode": img_mode,
                "width": width,
                "height": height,
                "dimension": f"{width}x{height}",
                "size_bytes": file_size_bytes,
                "size_kb": round(file_size_bytes / 1024, 2)
            })
        except Exception as e:
            flagged_files.append({
                "path": str(file_path),
                "class": class_name,
                "issue": f"Corrupted/unreadable ({type(e).__name__}: {e})"
            })

df_images = pd.DataFrame(records)
print(f"Scan finished: {len(records)} valid images, {len(flagged_files)} flagged files.")

### Integrity Summary
Summary reports for:
1. Total image count per class
2. Corrupted or unreadable files (listed explicitly if found)
3. Image dimension consistency and formats

In [ ]:
print("=" * 60)
print("1. TOTAL IMAGE COUNT PER CLASS")
print("=" * 60)
class_counts = df_images["class_name"].value_counts().sort_index()
for class_name, count in class_counts.items():
    print(f"  - {class_name:<28}: {count:,} images")
print(f"  {'Total':<28}: {len(df_images):,} images")

print("\n" + "=" * 60)
print("2. CORRUPTED / UNREADABLE / FLAGGED FILES")
print("=" * 60)
if flagged_files:
    print(f"WARNING: {len(flagged_files)} flagged file(s) found:")
    for f in flagged_files:
        print(f"  - File: {f['path']}")
        print(f"    Class: {f['class']}")
        print(f"    Issue: {f['issue']}")
else:
    print("  None! All image files opened and decoded successfully without corruption.")

print("\n" + "=" * 60)
print("3. IMAGE DIMENSION CONSISTENCY")
print("=" * 60)
dimension_counts = df_images["dimension"].value_counts()
print("Unique dimensions (Width x Height):")
for dim, count in dimension_counts.items():
    pct = (count / len(df_images)) * 100
    print(f"  - {dim:<15}: {count:,} images ({pct:.2f}%)")

if len(dimension_counts) == 1:
    print("\nResult: All images have identical dimensions across the dataset.")
else:
    print(f"\nResult: Dimension variation detected across {len(dimension_counts)} distinct sizes.")

print("\nFile Formats Detected:")
for fmt, count in df_images["format"].value_counts().items():
    print(f"  - {fmt}: {count:,} images")

## 3. Visual Inspection
For each of the 10 classes, randomly sample and display 5 real images in a horizontal grid with the class name as the title above each grid.

In [ ]:
SAMPLES_PER_CLASS = 5

for class_name in sorted(df_images["class_name"].unique()):
    class_df = df_images[df_images["class_name"] == class_name]
    sample_size = min(SAMPLES_PER_CLASS, len(class_df))
    sampled_records = class_df.sample(n=sample_size, random_state=RANDOM_SEED).to_dict("records")

    fig, axes = plt.subplots(1, sample_size, figsize=(16, 3.5))
    if sample_size == 1:
        axes = [axes]
    
    fig.suptitle(f"Class: {class_name} (Sample of {sample_size})", fontsize=14, fontweight="bold", y=1.05)

    for ax, rec in zip(axes, sampled_records):
        img = Image.open(rec["file_path"])
        ax.imshow(img)
        ax.set_title(f"{rec['filename']}\n{rec['width']}x{rec['height']} | {rec['size_kb']} KB", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 4. Summary of Findings

- 10,407 images across 10 classes. 0 corrupted files, 0 zero-byte files, 0 extension/format mismatches. All images are JPEG, RGB.
- Dimensions: 10,403 images are 480x640 (portrait). 4 are 640x480 (landscape): bacterial_leaf_blight/100622.jpg, bacterial_leaf_blight/103734.jpg, brown_spot/103050.jpg, brown_spot/103343.jpg. EXIF metadata is stripped across the entire dataset, so these are ordinary landscape photos, not a different source. Handle with an aspect-preserving resize at preprocessing, not a stretch.
- Class counts: normal 1,764; blast 1,738; hispa 1,594; dead_heart 1,442; tungro 1,088; brown_spot 965; downy_mildew 620; bacterial_leaf_blight 479; bacterial_leaf_streak 380; bacterial_panicle_blight 337. Imbalance is about 5.2:1.
- Visual inspection: real field photos with muddy water, weeds and reflections in the background; close-ups and wide shots are mixed; some images have strong glare. Tungro yellowing can resemble nutrient deficiency. Bacterial leaf blight, bacterial leaf streak and blast share brown necrotic streaks.
- Decision: use stratified splitting. Class weighting and minority augmentation are deferred until a baseline shows how much the imbalance hurts.


## 5. Metadata Analysis (Variety & Age from train.csv)

Load the dataset metadata from `data/raw/paddy-disease-classification/train.csv` and inspect:
1. Basic shape, integrity verification against the 10,407 image count, and null values.
2. Paddy variety distribution (unique counts and proportions).
3. Crop age distribution (unique values, min/max range, and frequency).
4. Joint `(variety, age)` grouping and group size distribution (evaluating its validity as a field session proxy).
5. Cross-tabulation of disease label vs. variety.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Resolve train.csv path (supports running from notebooks/ or project root)
csv_candidates = [
    Path("../data/raw/paddy-disease-classification/train.csv"),
    Path("data/raw/paddy-disease-classification/train.csv"),
]
train_csv_path = None
for p in csv_candidates:
    if p.exists():
        train_csv_path = p.resolve()
        break

if train_csv_path is None:
    raise FileNotFoundError("Could not locate 'train.csv'.")

df_train = pd.read_csv(train_csv_path)

# 1. Basic Shape and Image Count Verification
print("=" * 70)
print("1. BASIC SHAPE & ROW COUNT INTEGRITY")
print("=" * 70)
print(f"train.csv path:   {train_csv_path}")
print(f"DataFrame shape:  {df_train.shape} (Rows: {len(df_train):,}, Columns: {list(df_train.columns)})")

IMAGE_COUNT_EXPECTED = 10407
if len(df_train) == IMAGE_COUNT_EXPECTED:
    print(f"CONFIRMED: Row count ({len(df_train):,}) matches the expected {IMAGE_COUNT_EXPECTED:,} image count.")
else:
    print(f"FLAG / WARNING: Row count ({len(df_train):,}) DOES NOT MATCH the {IMAGE_COUNT_EXPECTED:,} image count!")

print("\nMissing values per column:")
print(df_train.isnull().sum())

# 2. Variety Distribution
print("\n" + "=" * 70)
print("2. PADDY VARIETY DISTRIBUTION")
print("=" * 70)
unique_varieties = df_train["variety"].nunique()
print(f"Number of unique paddy varieties: {unique_varieties}")

variety_counts = df_train["variety"].value_counts()
variety_pct = (df_train["variety"].value_counts(normalize=True) * 100).round(2)
df_variety = pd.DataFrame({"Image Count": variety_counts, "Percentage (%)": variety_pct})
display(df_variety)

# 3. Age Distribution
print("\n" + "=" * 70)
print("3. CROP AGE DISTRIBUTION")
print("=" * 70)
unique_ages = df_train["age"].nunique()
min_age = df_train["age"].min()
max_age = df_train["age"].max()
print(f"Number of unique crop age values: {unique_ages}")
print(f"Crop age range: Min = {min_age} days, Max = {max_age} days")

age_counts = df_train["age"].value_counts().sort_index()
age_pct = (df_train["age"].value_counts(normalize=True).sort_index() * 100).round(2)
df_age = pd.DataFrame({"Image Count": age_counts, "Percentage (%)": age_pct})
display(df_age)

# 4. (Variety, Age) Grouping as Field Session Proxy
print("\n" + "=" * 70)
print("4. (VARIETY, AGE) GROUPING AS SESSION / METADATA PROXY")
print("=" * 70)
group_counts = df_train.groupby(["variety", "age"]).size()
num_unique_combos = len(group_counts)
print(f"Number of unique (variety, age) combinations: {num_unique_combos}")
print(f"Group size summary statistics:")
print(f"  - Min:    {group_counts.min()}")
print(f"  - Median: {group_counts.median():.1f}")
print(f"  - Mean:   {group_counts.mean():.1f}")
print(f"  - Max:    {group_counts.max()}")

bins = [0, 1, 10, 100, np.inf]
bin_labels = ["1 image (Singletons)", "2-10 images", "11-100 images", "100+ images"]
binned_sizes = pd.cut(group_counts, bins=bins, labels=bin_labels, right=True)
group_size_dist = binned_sizes.value_counts().sort_index()
images_per_bin = group_counts.groupby(binned_sizes, observed=False).sum()

df_group_bins = pd.DataFrame({
    "Number of Combos": group_size_dist,
    "Combo %": (group_size_dist / num_unique_combos * 100).round(2),
    "Total Images Covered": images_per_bin,
    "Image %": (images_per_bin / len(df_train) * 100).round(2)
})
print("\nDistribution of Group Sizes across (Variety, Age) combinations:")
display(df_group_bins)

# 5. Cross-Tabulation: Label vs. Variety
print("\n" + "=" * 70)
print("5. CROSS-TABULATION: LABEL VS. VARIETY")
print("=" * 70)
crosstab = pd.crosstab(df_train["label"], df_train["variety"], margins=True)
print("Cross-tabulation (Counts):")
display(crosstab)

classes_per_variety = (crosstab.iloc[:-1, :-1] > 0).sum(axis=1)
print("\nNumber of varieties each disease class appears across:")
for cls, n_vars in classes_per_variety.items():
    print(f"  - {cls:<28}: appears in {n_vars:>2} / {unique_varieties} varieties")

varieties_per_class = (crosstab.iloc[:-1, :-1] > 0).sum(axis=0)
print("\nNumber of disease classes present in each variety:")
for var, n_classes in varieties_per_class.items():
    print(f"  - {var:<20}: contains {n_classes:>2} / 10 classes")

zero_count = (crosstab.iloc[:-1, :-1] == 0).sum().sum()
total_cells = crosstab.iloc[:-1, :-1].size
print(f"\nZero-count cells (disease absent from variety): {zero_count} / {total_cells} ({zero_count / total_cells * 100:.1f}%)")


- train.csv has 10,407 rows (matches the image count) and no missing values. Columns: image_id, label, variety, age.
- 10 varieties; ADT45 is 6,992 images (67.19%). 18 distinct ages, from 45 to 82 days.
- (variety, age) gives 41 combinations: min 5, median 112, mean 253.8, max 2,371 (ADT45, 70 days). 23 combinations with 100+ images cover 92.27% of the dataset. This is far too coarse to act as a field-session proxy, so it is rejected for grouping.
- Label x variety confound: normal appears in only 4 of 10 varieties; bacterial_panicle_blight in 3; 41 of 100 label-by-variety cells are empty; RR (36 images) contains only dead_heart and normal; Surya (32 images) is mostly downy_mildew. Variety-linked leaf appearance could act as a shortcut for the label, and a random split cannot detect it.
- Planned handling: keep the natural variety mix in the main split; hold out the Ponni variety as a separate stress set; report worst-group accuracy; never use variety as an input to the deployed model.
